In [1]:
from pathlib import Path
import json
import textwrap
import gzip
DATA_PATH = Path("/home/dstartsev/dataset/HumanEval/human-eval/data") / "HumanEval.jsonl.gz"

tasks = []
with gzip.open(DATA_PATH, "rt", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        tasks.append(json.loads(line))

print("Total tasks:", len(tasks))
print("Example of fields:", tasks[0].keys())
print("The first task_id:", tasks[0]["task_id"])

Total tasks: 164
Example of fields: dict_keys(['task_id', 'prompt', 'entry_point', 'canonical_solution', 'test'])
The first task_id: HumanEval/0


In [2]:
from transformers import AutoModel, AutoTokenizer
import torch
import re
from tqdm.auto import tqdm
import torch

/home/dstartsev/.conda/envs/hf-download/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
REPO_ID = "/home/dstartsev/models/dream-coder-7b"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

tokenizer = AutoTokenizer.from_pretrained(
    REPO_ID,
    trust_remote_code=True,
)

model = AutoModel.from_pretrained(
    REPO_ID,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    device_map="auto",
)

model.eval()   

Device: cuda


Loading checkpoint shards: 100%|██████████| 4/4 [02:34<00:00, 38.68s/it]


DreamModel(
  (model): DreamBaseModel(
    (embed_tokens): Embedding(152064, 3584, padding_idx=151643)
    (layers): ModuleList(
      (0-27): 28 x DreamDecoderLayer(
        (self_attn): DreamSdpaAttention(
          (q_proj): Linear(in_features=3584, out_features=3584, bias=True)
          (k_proj): Linear(in_features=3584, out_features=512, bias=True)
          (v_proj): Linear(in_features=3584, out_features=512, bias=True)
          (o_proj): Linear(in_features=3584, out_features=3584, bias=False)
          (rotary_emb): DreamRotaryEmbedding()
        )
        (mlp): DreamMLP(
          (gate_proj): Linear(in_features=3584, out_features=18944, bias=False)
          (up_proj): Linear(in_features=3584, out_features=18944, bias=False)
          (down_proj): Linear(in_features=18944, out_features=3584, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): DreamRMSNorm((3584,), eps=1e-06)
        (post_attention_layernorm): DreamRMSNorm((3584,), eps=1e-06)
      )

In [ ]:
TEMPERATURE = 0.1
TOP_P = 0.95
ALG_TEMP = 0.0  
EOS_PENALTY = 3.0
results = []

In [5]:
import ast
import textwrap

In [6]:
def clean_and_check(code: str):
    code = textwrap.dedent(code).strip()
    lines = code.splitlines()
    start = None
    for i, line in enumerate(lines):
        if line.lstrip().startswith("def "):
            start = i
            break
    if start is not None:
        code = "\n".join(lines[start:])
    try:
        ast.parse(code)
        return code, True
    except SyntaxError:
        return code, False

In [7]:


def make_humaneval_prompt(prompt_text: str):
    messages = [{"role": "user", "content": prompt_text.strip()}]
    inputs = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        return_dict=True,
        add_generation_prompt=True,
    )
    return inputs

In [8]:
def extract_first_function(text: str) -> str:
    lines = text.splitlines()

    start = None
    for i, line in enumerate(lines):
        if line.lstrip().startswith("def "):
            start = i
            break
    if start is None:
        return text  

    func_lines = [lines[start]]
    base_indent = len(lines[start]) - len(lines[start].lstrip())


    for line in lines[start+1:]:

        if line.strip() == "":
            func_lines.append(line)
            continue

        indent = len(line) - len(line.lstrip())
        if indent > base_indent:
            func_lines.append(line)
        else:
            
            break

    return "\n".join(func_lines)

In [10]:
from typing import Dict, Any
import signal

def run_tests_for_task(task: Dict[str, Any], gen_code: str) -> bool:
    prompt = task["prompt"]
    tests = task["test"]
    entry_point = task["entry_point"]
    full_src = prompt + "\n" + gen_code + "\n" + tests
    
    ns: Dict[str, Any] = {}
    try:
        exec(textwrap.dedent(full_src), ns)
    except Exception as e:
        print(f"[EXEC ERROR] {task['task_id']}: {e}")
        return False
    
    if "check" not in ns or entry_point not in ns:
        print(f"[NS MISSING] {task['task_id']}: 'check' or '{entry_point}' not in namespace")
        return False
    
    try:
        signal.alarm(3)
        ns["check"](ns[entry_point])
        signal.alarm(0)
        return True
    except Exception as e:
        signal.alarm(0)
        print(f"[CHECK ERROR] {task['task_id']}: {e}")
        return False

In [ ]:

import ast
from tqdm.auto import tqdm
import pandas as pd
import time

steps_list = [128]
max_new_list = [ 512]

all_rows = []

def generate_with_dreamcoder_params(prompt: str, steps: int, max_new: int):
    inputs = make_humaneval_prompt(prompt)
    input_ids = inputs["input_ids"].to(model.device)
    attention_mask = inputs["attention_mask"].to(model.device)

    with torch.no_grad():
        out = model.diffusion_generate(
            input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new,
            steps=steps,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            alg="entropy",
            alg_temp=ALG_TEMP,
            output_history=False,
            return_dict_in_generate=True,
        )

    input_len = input_ids.shape[1]
    gen_ids = out.sequences[0, input_len:]
    gen_text = tokenizer.decode(gen_ids, skip_special_tokens=True)

    if tokenizer.eos_token is not None:
        gen_text = gen_text.split(tokenizer.eos_token)[0]

    gen_text = extract_first_function(gen_text)

    code, _ = clean_and_check(gen_text)
    return code, gen_ids.shape[-1]   

In [ ]:

checkpoint_path = "humaneval_dreamcoder_grid_checkpoint.csv"
per_sample_path = "humaneval_dreamcoder_samples.jsonl"  
task_by_id = {t["task_id"]: t for t in tasks}

for steps in steps_list:
    for max_new in max_new_list:
        print(f"\n=== STEPS = {steps}, MAX_NEW = {max_new} ===")

        results = []
        n_tasks = 0
        n_pass = 0
        n_ast_ok = 0
        total_len_tokens = 0
        n_trunc = 0

        t_start = time.time()

        for task in tqdm(tasks):  
            n_tasks += 1
            task_id = task["task_id"]
            prompt = task["prompt"]

            gen_code, gen_len_tokens = generate_with_dreamcoder_params(prompt, steps, max_new)

            try:
                ast.parse(gen_code)
                ast_ok = True
            except SyntaxError as e:
                print(f"[AST ERROR] {task_id} @ steps={steps}, max_new={max_new}: {e}")
                ast_ok = False

            if ast_ok:
                n_ast_ok += 1

            if gen_len_tokens >= max_new:
                n_trunc += 1

            total_len_tokens += gen_len_tokens

            passed = run_tests_for_task(task, gen_code) if ast_ok else False
            if passed:
                n_pass += 1

            results.append({
                "task_id": task_id,
                "gen_code": gen_code,
                "ast_ok": ast_ok,
                "passed": passed,
                "gen_len_tokens": gen_len_tokens,
            })

            print(f"{task_id}: {'Correct' if passed else 'Incorrect'}")

            with open(per_sample_path, "a", encoding="utf-8") as f:
                f.write(json.dumps({
                    "steps": steps,
                    "max_new_tokens": max_new,
                    "task_id": task_id,
                    "ast_ok": ast_ok,
                    "passed": passed,
                    "gen_len_tokens": gen_len_tokens,
                }, ensure_ascii=False) + "\n")

        t_end = time.time()
        wall_time = t_end - t_start

        pass_at_1 = n_pass / n_tasks if n_tasks > 0 else 0.0
        ast_validity = n_ast_ok / n_tasks if n_tasks > 0 else 0.0
        trunc_rate = n_trunc / n_tasks if n_tasks > 0 else 0.0
        avg_len_tokens = total_len_tokens / n_tasks if n_tasks > 0 else 0.0
        avg_time_per_task = wall_time / n_tasks if n_tasks > 0 else 0.0

        print(
            f"RESULTS steps={steps}, max_new={max_new}: "
            f"Pass@1={pass_at_1:.3f}, AST-validity={ast_validity:.3f}, "
            f"Trunc={trunc_rate:.3f}, avg_len={avg_len_tokens:.1f}, "
            f"time/task={avg_time_per_task:.2f}s"
        )

        row = {
            "steps": steps,
            "max_new_tokens": max_new,
            "num_tasks": n_tasks,
            "pass_at_1": pass_at_1,
            "ast_validity": ast_validity,
            "truncation_rate": trunc_rate,
            "avg_len_tokens": avg_len_tokens,
            "avg_time_per_task_sec": avg_time_per_task,
            "total_time_sec": wall_time,
        }
        all_rows.append(row)


        df_chk = pd.DataFrame(all_rows)
        df_chk.to_csv(checkpoint_path, index=False)
        print(f"Checkpoint saved to {checkpoint_path}")


final_path = "humaneval_dreamcoder_grid_final.csv"
df = pd.DataFrame(all_rows)
df.to_csv(final_path, index=False)
print(f"\nFinal metrics saved to {final_path}")


=== STEPS = 128, MAX_NEW = 512 ===


  1%|          | 1/164 [00:24<1:07:31, 24.85s/it]

[AST ERROR] HumanEval/0 @ steps=128, max_new=512: invalid syntax (<unknown>, line 10)
HumanEval/0: Incorrect


  1%|          | 2/164 [00:45<1:00:49, 22.53s/it]

[AST ERROR] HumanEval/1 @ steps=128, max_new=512: unterminated string literal (detected at line 7) (<unknown>, line 7)
HumanEval/1: Incorrect


...

[AST ERROR] HumanEval/41 @ steps=128, max_new=512: expected ':' (<unknown>, line 1)
HumanEval/41: Incorrect


 26%|██▌       | 43/164 [15:46<45:40, 22.65s/it]

HumanEval/42: Correct


 27%|██▋       | 44/164 [16:11<46:19, 23.16s/it]

[CHECK ERROR] HumanEval/43: 
HumanEval/43: Incorrect


In [13]:
import ast
from tqdm.auto import tqdm
import pandas as pd
import time
import json

steps_list = [ 256, 512]
max_new_list = [256, 512]

chunk_size = 40  # по 40 задач в чанке
all_rows = []
per_sample_path = "humaneval_dreamcoder_samples_chunks.jsonl"
checkpoint_path = "humaneval_dreamcoder_grid_chunks.csv"

def iter_chunks(lst, size):
    for i in range(0, len(lst), size):
        yield i, lst[i:i+size]

for steps in steps_list:
    for max_new in max_new_list:
        print(f"\n=== STEPS = {steps}, MAX_NEW = {max_new} ===")

        for chunk_idx, tasks_chunk in iter_chunks(tasks, chunk_size):
            print(f"\n--- CHUNK {chunk_idx}..{chunk_idx+len(tasks_chunk)-1} ---")

            n_tasks = 0
            n_pass = 0
            n_ast_ok = 0
            total_len_tokens = 0
            n_trunc = 0

            t_start = time.time()

            for task in tqdm(tasks_chunk):
                n_tasks += 1
                task_id = task["task_id"]
                prompt = task["prompt"]

                gen_code, gen_len_tokens = generate_with_dreamcoder_params(
                    prompt, steps, max_new
                )

                try:
                    ast.parse(gen_code)
                    ast_ok = True
                except SyntaxError as e:
                    print(f"[AST ERROR] {task_id} @ steps={steps}, max_new={max_new}: {e}")
                    ast_ok = False

                if ast_ok:
                    n_ast_ok += 1
                if gen_len_tokens >= max_new:
                    n_trunc += 1

                total_len_tokens += gen_len_tokens

                passed = run_tests_for_task(task, gen_code) if ast_ok else False
                if passed:
                    n_pass += 1

                print(f"{task_id}: {'Correct' if passed else 'Incorrect'}")

                # пишем построчно, чтобы не потерять этот пример
                with open(per_sample_path, "a", encoding="utf-8") as f:
                    f.write(json.dumps({
                        "steps": steps,
                        "max_new_tokens": max_new,
                        "chunk_start": chunk_idx,
                        "task_id": task_id,
                        "ast_ok": ast_ok,
                        "passed": passed,
                        "gen_len_tokens": gen_len_tokens,
                    }, ensure_ascii=False) + "\n")

            t_end = time.time()
            wall_time = t_end - t_start

            pass_at_1 = n_pass / n_tasks if n_tasks > 0 else 0.0
            ast_validity = n_ast_ok / n_tasks if n_tasks > 0 else 0.0
            trunc_rate = n_trunc / n_tasks if n_tasks > 0 else 0.0
            avg_len_tokens = total_len_tokens / n_tasks if n_tasks > 0 else 0.0
            avg_time_per_task = wall_time / n_tasks if n_tasks > 0 else 0.0

            print(
                f"CHUNK {chunk_idx}: Pass@1={pass_at_1:.3f}, "
                f"AST-validity={ast_validity:.3f}, Trunc={trunc_rate:.3f}, "
                f"avg_len={avg_len_tokens:.1f}, time/task={avg_time_per_task:.2f}s"
            )

            all_rows.append({
                "steps": steps,
                "max_new_tokens": max_new,
                "chunk_start": chunk_idx,
                "chunk_size": n_tasks,
                "pass_at_1": pass_at_1,
                "ast_validity": ast_validity,
                "truncation_rate": trunc_rate,
                "avg_len_tokens": avg_len_tokens,
                "avg_time_per_task_sec": avg_time_per_task,
                "total_time_sec": wall_time,
            })

            # чекпоинт после КАЖДОГО чанка
            pd.DataFrame(all_rows).to_csv(checkpoint_path, index=False)
            print(f"Checkpoint saved to {checkpoint_path}")


=== STEPS = 256, MAX_NEW = 256 ===

--- CHUNK 0..39 ---


  2%|▎         | 1/40 [00:30<19:46, 30.42s/it]

[CHECK ERROR] HumanEval/0: 
HumanEval/0: Incorrect


  5%|▌         | 2/40 [00:57<17:57, 28.36s/it]

[CHECK ERROR] HumanEval/1: 
HumanEval/1: Incorrect


...

HumanEval/79: Correct
CHUNK 40: Pass@1=0.650, AST-validity=0.825, Trunc=1.000, avg_len=512.0, time/task=45.64s
Checkpoint saved to humaneval_dreamcoder_grid_chunks.csv

--- CHUNK 80..119 ---


  2%|▎         | 1/40 [00:47<31:06, 47.85s/it]

HumanEval/80: Correct


  2%|▎         | 1/40 [01:36<1:02:46, 96.57s/it]


KeyboardInterrupt: 

In [18]:
checkpoint_path = "humaneval_dreamcoder_grid_chunks1.csv"
per_sample_path = "humaneval_dreamcoder_samples.jsonl"  # опционально
task_by_id = {t["task_id"]: t for t in tasks}
steps_list = [ 256, 512]
max_new_list = [ 512]

import ast
from tqdm.auto import tqdm
import pandas as pd
import time
import json

def iter_chunks(lst, size):
    for i in range(0, len(lst), size):
        yield i, lst[i:i+size]
for steps in steps_list:
    for max_new in max_new_list:
        print(f"\n=== STEPS = {steps}, MAX_NEW = {max_new} ===")

        results = []
        n_tasks = 0
        n_pass = 0
        n_ast_ok = 0
        total_len_tokens = 0
        n_trunc = 0

        t_start = time.time()

        for task in tqdm(tasks[:40]):
            n_tasks += 1
            task_id = task["task_id"]
            prompt = task["prompt"]

            gen_code, gen_len_tokens = generate_with_dreamcoder_params(prompt, steps, max_new)

            try:
                ast.parse(gen_code)
                ast_ok = True
            except SyntaxError as e:
                print(f"[AST ERROR] {task_id} @ steps={steps}, max_new={max_new}: {e}")
                ast_ok = False

            if ast_ok:
                n_ast_ok += 1

            if gen_len_tokens >= max_new:
                n_trunc += 1

            total_len_tokens += gen_len_tokens

            passed = run_tests_for_task(task, gen_code) if ast_ok else False
            if passed:
                n_pass += 1

            results.append({
                "task_id": task_id,
                "gen_code": gen_code,
                "ast_ok": ast_ok,
                "passed": passed,
                "gen_len_tokens": gen_len_tokens,
            })

            print(f"{task_id}: {'Correct' if passed else 'Incorrect'}")

            with open(per_sample_path, "a", encoding="utf-8") as f:
                f.write(json.dumps({
                    "steps": steps,
                    "max_new_tokens": max_new,
                    "task_id": task_id,
                    "ast_ok": ast_ok,
                    "passed": passed,
                    "gen_len_tokens": gen_len_tokens,
                }, ensure_ascii=False) + "\n")

        t_end = time.time()
        wall_time = t_end - t_start

        pass_at_1 = n_pass / n_tasks if n_tasks > 0 else 0.0
        ast_validity = n_ast_ok / n_tasks if n_tasks > 0 else 0.0
        trunc_rate = n_trunc / n_tasks if n_tasks > 0 else 0.0
        avg_len_tokens = total_len_tokens / n_tasks if n_tasks > 0 else 0.0
        avg_time_per_task = wall_time / n_tasks if n_tasks > 0 else 0.0

        print(
            f"RESULTS steps={steps}, max_new={max_new}: "
            f"Pass@1={pass_at_1:.3f}, AST-validity={ast_validity:.3f}, "
            f"Trunc={trunc_rate:.3f}, avg_len={avg_len_tokens:.1f}, "
            f"time/task={avg_time_per_task:.2f}s"
        )

        row = {
            "steps": steps,
            "max_new_tokens": max_new,
            "num_tasks": n_tasks,
            "pass_at_1": pass_at_1,
            "ast_validity": ast_validity,
            "truncation_rate": trunc_rate,
            "avg_len_tokens": avg_len_tokens,
            "avg_time_per_task_sec": avg_time_per_task,
            "total_time_sec": wall_time,
        }
        all_rows.append(row)


        df_chk = pd.DataFrame(all_rows)
        df_chk.to_csv(checkpoint_path, index=False)
        print(f"Checkpoint saved to {checkpoint_path}")


final_path = "humaneval_dreamcoder_grid_final1.csv"
df = pd.DataFrame(all_rows)
df.to_csv(final_path, index=False)
print(f"\nFinal metrics saved to {final_path}")


=== STEPS = 256, MAX_NEW = 512 ===


  2%|▎         | 1/40 [00:48<31:17, 48.13s/it]

[AST ERROR] HumanEval/0 @ steps=256, max_new=512: unterminated string literal (detected at line 1) (<unknown>, line 1)
HumanEval/0: Incorrect


  5%|▌         | 2/40 [01:30<28:17, 44.68s/it]

[AST ERROR] HumanEval/1 @ steps=256, max_new=512: unindent does not match any outer indentation level (<unknown>, line 28)
HumanEval/1: Incorrect


...

HumanEval/37: Correct


 98%|█████████▊| 39/40 [57:43<01:33, 93.03s/it]

HumanEval/38: Correct


100%|██████████| 40/40 [59:08<00:00, 88.72s/it]

[CHECK ERROR] HumanEval/39: 
HumanEval/39: Incorrect
RESULTS steps=512, max_new=512: Pass@1=0.875, AST-validity=1.000, Trunc=1.000, avg_len=512.0, time/task=88.72s
Checkpoint saved to humaneval_dreamcoder_grid_chunks1.csv

Final metrics saved to humaneval_dreamcoder_grid_final1.csv
